# SHACL Validation

Generate shapes from models and validate data with pyshacl.

**Requires:** `pip install rdfantic[shacl]`

## Model with SHACL constraints

In [1]:
from typing import Annotated

from rdflib import RDF, XSD, Graph, Literal, Namespace

from rdfantic import GraphModel, SHConstraint, predicate

SCHEMA = Namespace("http://schema.org/")
EX = Namespace("http://example.org/")


class StrictPerson(GraphModel):
    rdf_type = SCHEMA["Person"]
    name: Annotated[
        str,
        SHConstraint(
            min_length=1,
            max_length=200,
            name="Full Name",
        ),
    ] = predicate(SCHEMA["name"])
    age: Annotated[
        int,
        SHConstraint(
            min_inclusive=0,
            max_inclusive=150,
        ),
    ] = predicate(SCHEMA["age"])

## Generate SHACL shape

In [2]:
shacl_graph = StrictPerson.to_shacl()

print("Generated SHACL shape:")
print(shacl_graph.serialize(format="turtle"))

Generated SHACL shape:
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

[] a sh:NodeShape ;
    sh:property [ sh:datatype xsd:string ;
            sh:maxCount 1 ;
            sh:maxLength 200 ;
            sh:minCount 1 ;
            sh:minLength 1 ;
            sh:name "Full Name" ;
            sh:path <http://schema.org/name> ],
        [ sh:datatype xsd:integer ;
            sh:maxCount 1 ;
            sh:maxInclusive 150 ;
            sh:minCount 1 ;
            sh:minInclusive 0 ;
            sh:path <http://schema.org/age> ] ;
    sh:targetClass <http://schema.org/Person> .




## Validate: valid data

In [3]:
valid = Graph()
valid.add((EX["alice"], RDF.type, SCHEMA["Person"]))
valid.add((EX["alice"], SCHEMA["name"], Literal("Alice", datatype=XSD.string)))
valid.add((EX["alice"], SCHEMA["age"], Literal(30, datatype=XSD.integer)))

from pyshacl import validate

conforms, _, report = validate(valid, shacl_graph=shacl_graph)
print(f"Valid data conforms: {conforms}")

Valid data conforms: True


## Validate: invalid data (age = -5)

In [4]:
invalid = Graph()
invalid.add((EX["bob"], RDF.type, SCHEMA["Person"]))
invalid.add((EX["bob"], SCHEMA["name"], Literal("Bob", datatype=XSD.string)))
invalid.add((EX["bob"], SCHEMA["age"], Literal(-5, datatype=XSD.integer)))

conforms, _, report = validate(invalid, shacl_graph=shacl_graph)
print(f"Invalid data conforms: {conforms}")
if not conforms:
    print(f"\nValidation report:\n{report}")

Invalid data conforms: False

Validation report:
Validation Report
Conforms: False
Results (1):
Constraint Violation in MinInclusiveConstraintComponent (http://www.w3.org/ns/shacl#MinInclusiveConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:datatype xsd:integer ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:maxInclusive Literal("150", datatype=xsd:integer) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:minInclusive Literal("0", datatype=xsd:integer) ; sh:path <http://schema.org/age> ]
	Focus Node: <http://example.org/bob>
	Value Node: Literal("-5", datatype=xsd:integer)
	Result Path: <http://schema.org/age>
	Message: Value is not >= Literal("0", datatype=xsd:integer)

